<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/11-catalyst-optimizer/00-plan-pipeline.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 11.0 — The plan pipeline: four trees on one object

See your DataFrame code turn into a tree, then watch that tree pass
through parsed → analyzed → optimized → physical. Every plan is just
an attribute on `df.queryExecution`; `explain()` only formats them.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("11.0-plan-pipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## A DataFrame call builds a tree — it does not run

`filter(...).select(...)` returns a new DataFrame wrapping a
`LogicalPlan`. Nothing has executed. Read the tree **bottom-up**:
relation → filter → project.

In [ ]:
df = spark.range(1_000_000).withColumn("k", (col("id") % 50).cast("int")) \
                             .withColumn("v", col("id") * 2)

plan = df.filter(col("v") > 100).select("k")
print(type(plan))                 # still a DataFrame — nothing ran
print(plan.queryExecution.logical)  # the raw (parsed) tree, bottom-up

## The four plans live on `queryExecution`

`explain()` is a pretty-printer over these four attributes. Look at
them as objects and the pipeline stops being magic.

In [ ]:
qe = df.filter(col("v") > 100).select("k").queryExecution

print("=== 1. parsed (unresolved) ===");   print(qe.logical)
print("\n=== 2. analyzed (typed, names bound) ==="); print(qe.analyzed)
print("\n=== 3. optimized logical ===");    print(qe.optimizedPlan)
print("\n=== 4. physical (executed) ===");  print(qe.executedPlan)

## Watch the optimizer actually change the tree

Diff analyzed vs optimized on a query with obvious slack: a redundant
filter and an unused column. The optimized tree is smaller.

In [ ]:
qe2 = (df
       .filter(col("v") > 100)
       .filter(F.lit(1) == F.lit(1))     # always true — optimizer should delete it
       .select("k")                       # 'v' and 'id' now unused
      ).queryExecution

print("ANALYZED (what you wrote):\n", qe2.analyzed)
print("\nOPTIMIZED (what Spark will run):\n", qe2.optimizedPlan)
# note: the always-true filter is gone; the projection narrowed

## Stage 2 is where typos die — cheaply, before any job runs

An unknown column raises `AnalysisException` at analysis time —
instantly, no cluster work. This is the *good* class of error.

In [ ]:
from pyspark.sql.utils import AnalysisException

try:
    df.select("amt").collect()      # column is 'v', not 'amt'
except AnalysisException as e:
    print("caught at ANALYSIS time (stage 2), no job ran:\n")
    print(str(e).splitlines()[0])

## SQL and the DataFrame API converge to the same tree

Register a view, express the same query both ways, compare optimized
plans. Identical tree ⇒ identical performance (Module 4's promise).

In [ ]:
df.createOrReplaceTempView("t")

sql_plan = spark.sql("SELECT k FROM t WHERE v > 100").queryExecution.optimizedPlan
api_plan = df.filter(col("v") > 100).select("k").queryExecution.optimizedPlan

print("SQL optimized:\n", sql_plan)
print("\nAPI optimized:\n", api_plan)
print("\nSame tree? ->", sql_plan.sameResult(api_plan))

In [ ]:
spark.stop()

## Exercises

1. Build `df.groupBy("k").sum("v")` and print all four plans off
   `queryExecution`. Which plan is the first to contain an
   `Exchange` node, and why does it not appear in the logical plans?
2. `df.select("k").filter(col("v") > 100)` — you wrote select *before*
   filter, but check `optimizedPlan`. Did the optimizer keep your
   order? What rule reordered it, and why is it allowed to?
3. Trigger three different `AnalysisException`s: unknown column,
   unknown function, and a type error (e.g. `col("k") + "abc"`
   somewhere invalid). Confirm each fails before a job starts.
4. Use `qe.optimizedPlan.sameResult(other)` to prove that
   `df.filter(A).filter(B)` and `df.filter(A & B)` optimize to the
   same tree.
5. Print `df.queryExecution.executedPlan` and count the `*(n)`
   prefixes. Peek ahead: what do you think they mean? (Lesson 11.3.)

In [ ]:
# your exercise code here